# BraTS 2021 Preprocessing Pipeline — All Fixes Applied
**SEIS 766 Vision AI — Spring 2026**

Corrected task ordering: `SEG-01 → SEG-02 → SEG-03 → SEG-06 → SEG-04 → SEG-05`

All fixes from code review applied:
1. SEG-01: Spot-check 3 random cases with shape verification
2. SEG-03: `np.nan_to_num()` before normalization
3. SEG-06: `.copy()` before shuffle to protect caller's list
4. SEG-06: Check if split JSON already exists
5. SEG-04: `+1` on end_z to include full margin
6. SEG-04: Partition-aware extraction (train/val saved separately)
7. SEG-05: `.copy()` after `np.flip` for array contiguity
8. SEG-05: Geometric and photometric transforms separated
9. SEG-05: `borderMode`/`borderValue` in warpAffine
10. SEG-05: Mask cast to float32 before warp, uint8 after
11. SEG-05: Wrapped in Dataset class (train augments, val doesn't)
12. Execution: leakage verification check


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/"


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SMART ROUTER: Detect which scenario applies and skip work
#
# This cell inspects Drive + local SSD to figure out the fastest
# path for your current situation, then tells you exactly which
# cells to run.
#
# Scenario A (first time ever, ~25-40 min):
#   - brats_slices.zip does NOT exist on shared Drive
#   - Run ALL cells below in order
#
# Scenario B (slices already extracted, ~2-5 min):
#   - brats_slices.zip EXISTS on shared Drive
#   - SKIP cells 3-10 entirely
#   - Run Cell 2 (imports), Cell 11 (load slices zip), Cell 12
#     (dataset wrapper), Cell 13 (create datasets)
#
# Scenario C (local slices still exist from prior session, instant):
#   - /content/local_slices/slices/train/ has .npz files
#   - Runtime didn't disconnect — SKIP everything preprocessing
#   - Go straight to Cell 12, Cell 13, then Phase 2
# ═══════════════════════════════════════════════════════════════

import os

# Paths (these get set properly in Cell 2, but we need them now)
_ROOT = '/content/drive/MyDrive/'
_SHARED = _ROOT + 'Group_Project_766'
_SLICES_ZIP = os.path.join(_SHARED, 'brats_slices.zip')
_RAW_ZIP = os.path.join(_SHARED, 'brats_raw.zip')
_LOCAL_SLICES = '/content/local_slices/slices/train'

has_local_slices = os.path.exists(_LOCAL_SLICES) and \
                   len([f for f in os.listdir(_LOCAL_SLICES) if f.endswith('.npz')]) > 100 \
                   if os.path.exists(_LOCAL_SLICES) else False
has_slices_zip = os.path.exists(_SLICES_ZIP)
has_raw_zip = os.path.exists(_RAW_ZIP)

print('═' * 60)
print('ENVIRONMENT CHECK')
print('═' * 60)
print(f'Local extracted slices (this session): {"✅ YES" if has_local_slices else "❌ no"}')
print(f'brats_slices.zip on shared Drive:      {"✅ YES" if has_slices_zip else "❌ no"}')
print(f'brats_raw.zip on shared Drive:         {"✅ YES" if has_raw_zip else "❌ no"}')
print()

if has_local_slices:
    print('🚀 SCENARIO C — Local slices already extracted (instant)')
    print()
    print('   WHAT TO RUN:')
    print('   → Cell 2    (imports + paths)')
    print('   → Cell 12   (SEG-05 augmentation functions)')
    print('   → Cell 13   (create datasets)')
    print('   → Phase 2 cells (starting at Cell 15)')
    print()
    print('   SKIP cells 3, 4, 5, 6, 7, 8, 9, 10, 11 — not needed.')

elif has_slices_zip:
    print('⚡ SCENARIO B — Slices zip on Drive (2-5 min to unzip)')
    print()
    print('   WHAT TO RUN:')
    print('   → Cell 2    (imports + paths)')
    print('   → Cell 11   (unzip brats_slices.zip to local SSD)')
    print('   → Cell 12   (SEG-05 augmentation functions)')
    print('   → Cell 13   (create datasets)')
    print('   → Phase 2 cells (starting at Cell 15)')
    print()
    print('   SKIP cells 3, 4, 5, 6, 7, 8, 9, 10 — slices already done.')

elif has_raw_zip:
    print('🏗️  SCENARIO A — First-time full extraction (25-40 min)')
    print()
    print('   WHAT TO RUN (in order):')
    print('   → Cell 2    (imports + paths)')
    print('   → Cell 3    (unzip raw BraTS from Drive, ~5-10 min)')
    print('   → Cells 4-7 (SEG-01 through SEG-06)')
    print('   → Cell 8    (define parallel extraction functions)')
    print('   → Cell 9    (run parallel extraction, ~15-20 min)')
    print('   → Cell 10   (zip slices + upload to Drive, ~5-10 min)')
    print('   → Cell 12   (SEG-05 augmentation functions)')
    print('   → Cell 13   (create datasets)')
    print('   → Phase 2 cells (starting at Cell 15)')
    print()
    print('   SKIP cell 11 — no slices zip exists yet (you\'ll create it in Cell 10).')

else:
    print('❌ NO DATA FOUND')
    print()
    print('   No raw zip, no slices zip, no local slices.')
    print('   Check that the shared Drive folder is mounted correctly.')
    print(f'   Expected: {_SHARED}')
    print(f'   Exists:   {os.path.exists(_SHARED)}')

print()
print('═' * 60)


In [ ]:
import os
import json
import random
import time
import shutil
import zipfile
import numpy as np
import cv2
import nibabel as nib
from concurrent.futures import ProcessPoolExecutor, as_completed

# ═══════════════════════════════════════════════════════════════
# PATHS
#
# SHARED_FOLDER: The shared Drive folder (from an external account)
#   containing brats_raw.zip and where results go.
# LOCAL_DATA:    Local SSD copy of raw BraTS data (fast read)
# LOCAL_DIR:     Local SSD for extracted slices (fast write)
# OUTPUT_DIR:    Persistent Drive storage for split JSON, model
#                checkpoints, and slice zip.
#
# STRATEGY: Unzip raw BraTS to local SSD ONCE, process there,
# then save only the slice zip + model artifacts back to Drive.
#
# NOTE ON SHARED FOLDER: Because the Group_Project_766 folder is
# shared from another Colab account, we have READ-ONLY access by
# default. Write operations go to YOUR MyDrive, not the shared
# folder. If you need the slice zip written to the SHARED folder
# (so teammates can use it), the owner of the shared folder must
# grant you Editor access in Drive sharing settings.
# ═══════════════════════════════════════════════════════════════

# Path to the shared Drive folder (read-only unless granted Editor)
SHARED_FOLDER = ROOT + "Group_Project_766"

# Source of raw BraTS data (zip in the shared folder)
SHARED_RAW_ZIP = os.path.join(SHARED_FOLDER, "brats_raw.zip")

# Optional: If teammates have already extracted slices and the
# slice zip exists in the shared folder, we can unzip directly
# instead of re-running the raw extraction.
SHARED_SLICES_ZIP = os.path.join(SHARED_FOLDER, "brats_slices.zip")

# Local SSD working directories
LOCAL_DATA = "/content/brats_raw"         # unzipped raw data lives here
LOCAL_DIR  = "/content/local_slices"      # extracted slices live here

# Where OUR outputs go (checkpoints, results, etc.)
# This is in the shared folder if you have Editor access, otherwise
# change to a path in your personal MyDrive.
OUTPUT_DIR = SHARED_FOLDER

# Raw data path (used by SEG-01 through SEG-04 for loading)
TRAIN_PATH = LOCAL_DATA

# Sanity checks
print(f"Shared folder exists:  {os.path.exists(SHARED_FOLDER)}")
print(f"Raw zip exists:        {os.path.exists(SHARED_RAW_ZIP)}")
print(f"Slices zip exists:     {os.path.exists(SHARED_SLICES_ZIP)}")

# Check write access to OUTPUT_DIR
try:
    test_file = os.path.join(OUTPUT_DIR, '.write_test.tmp')
    with open(test_file, 'w') as f:
        f.write('test')
    os.remove(test_file)
    print(f"Write access to OUTPUT_DIR: ✅ yes")
except (OSError, PermissionError) as e:
    print(f"Write access to OUTPUT_DIR: ❌ NO ({e})")
    print(f"   Change OUTPUT_DIR to your personal MyDrive path.")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Unzip Raw BraTS Data from Shared Drive → Local SSD
#
# SMART LOGIC:
# 1. If slices zip exists on Drive, skip raw data entirely
#    (run the 'Load Slices from Drive Zip' cell instead).
# 2. If local raw data already exists, skip unzip.
# 3. Otherwise, unzip from the shared Drive folder.
#
# HANDLES NESTED ZIP STRUCTURE:
# Some zips contain BraTS2021_* directly at root, others contain
# an intermediate folder like 'brats_raw/BraTS2021_*'. This cell
# detects both and points TRAIN_PATH to the correct level.
# ═══════════════════════════════════════════════════════════════

def find_brats_root(search_dir):
    """Find the directory that directly contains BraTS2021_* folders."""
    for root, dirs, _ in os.walk(search_dir):
        brats_dirs = [d for d in dirs if d.startswith('BraTS2021')]
        if len(brats_dirs) > 100:  # need substantial count to confirm
            return root
    return None

# Check if we can skip raw data entirely (slices already exist)
if os.path.exists(SHARED_SLICES_ZIP):
    print(f"ℹ️  Slices zip detected at {SHARED_SLICES_ZIP}")
    print(f"   You can SKIP this cell and SKIP extraction cells.")
    print(f"   Run the 'Load Slices from Drive Zip' cell instead.")

# Check if local raw data already exists
existing_root = find_brats_root(LOCAL_DATA) if os.path.exists(LOCAL_DATA) else None

if existing_root:
    count = len([d for d in os.listdir(existing_root) if d.startswith('BraTS2021')])
    print(f"✅ Local raw data already exists: {count} cases in {existing_root}")
    TRAIN_PATH = existing_root  # update in case it's nested
elif not os.path.exists(SHARED_RAW_ZIP):
    print(f"❌ Raw zip not found at {SHARED_RAW_ZIP}")
    print(f"   Cannot proceed without raw data or slice zip.")
else:
    print(f"📥 Unzipping raw BraTS data from shared Drive...")
    print(f"   From: {SHARED_RAW_ZIP}")
    print(f"   To:   {LOCAL_DATA}")
    print(f"   (Expected: 5-10 minutes)")

    start = time.time()
    os.makedirs(LOCAL_DATA, exist_ok=True)

    # -q = quiet, -o = overwrite existing (important if a previous
    #  partial unzip left fragments). Using !unzip instead of Python's
    # zipfile because it's faster for large archives.
    !unzip -q -o "{SHARED_RAW_ZIP}" -d "{LOCAL_DATA}/"

    elapsed = time.time() - start

    # Detect the correct BraTS root directory (handles nested structure)
    actual_root = find_brats_root(LOCAL_DATA)
    if actual_root is None:
        print(f"❌ Unzip completed but no BraTS2021_* folders found!")
        print(f"   Check zip structure with: !ls {LOCAL_DATA}")
    else:
        TRAIN_PATH = actual_root  # update path to the real root
        count = len([d for d in os.listdir(actual_root) if d.startswith('BraTS2021')])
        print(f"\n✅ Unzipped {count} cases in {elapsed/60:.1f} minutes")
        if actual_root != LOCAL_DATA:
            print(f"   (BraTS root is nested at: {actual_root})")
        print(f"   TRAIN_PATH is now: {TRAIN_PATH}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-01: Verify BraTS 2021 Dataset (reads from local SSD)
# ═══════════════════════════════════════════════════════════════

def verify_data():
    if not os.path.exists(TRAIN_PATH):
        print(f"❌ Folder not found at: {TRAIN_PATH}")
        return []

    cases = sorted([d for d in os.listdir(TRAIN_PATH)
                    if d.startswith('BraTS2021') and
                    os.path.isdir(os.path.join(TRAIN_PATH, d))])
    print(f"📂 Found {len(cases)} training cases in {TRAIN_PATH}")

    if cases:
        spot_checks = random.sample(cases, min(3, len(cases)))
        suffixes = ['_t1.nii.gz','_t1ce.nii.gz','_t2.nii.gz','_flair.nii.gz','_seg.nii.gz']
        for cid in spot_checks:
            for s in suffixes:
                fp = os.path.join(TRAIN_PATH, cid, f"{cid}{s}")
                if not os.path.exists(fp):
                    print(f"  ⚠️ Missing: {fp}")
                else:
                    img = nib.load(fp)
                    ok = '✅' if img.shape == (240,240,155) else '⚠️'
                    print(f"  {ok} {cid}{s} → {img.shape}")
    return cases

all_cases = verify_data()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-02: NIfTI Loading Pipeline
# ═══════════════════════════════════════════════════════════════

def load_case_data(case_id, data_path=None):
    """Load all 4 modalities + seg mask for one case."""
    if data_path is None:
        data_path = TRAIN_PATH
    case_path = os.path.join(data_path, case_id)
    data_dict = {}
    for mod in ['t1', 't1ce', 't2', 'flair']:
        fp = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
        data_dict[mod] = nib.load(fp).get_fdata().astype(np.float32)
    seg_fp = os.path.join(case_path, f"{case_id}_seg.nii.gz")
    data_dict['seg'] = nib.load(seg_fp).get_fdata().astype(np.uint8)
    return data_dict

if all_cases:
    s = load_case_data(all_cases[0])
    print(f"✅ Loaded {all_cases[0]}: " +
          ', '.join(f"{k}={v.shape}" for k,v in s.items()))


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-03: Per-Volume Z-Score Normalization
# FIX: np.nan_to_num() before computing statistics
# ═══════════════════════════════════════════════════════════════

def normalize_volume(volume):
    volume = np.nan_to_num(volume, nan=0.0, posinf=0.0, neginf=0.0)
    brain_mask = volume > 0
    if np.any(brain_mask):
        mean = volume[brain_mask].mean()
        std = volume[brain_mask].std()
        normalized = (volume - mean) / (std + 1e-8)
        normalized[~brain_mask] = 0
        return normalized.astype(np.float32)
    return volume.astype(np.float32)

def normalize_case(data_dict):
    for mod in ['t1', 't1ce', 't2', 'flair']:
        data_dict[mod] = normalize_volume(data_dict[mod])
    return data_dict

if all_cases:
    norm = normalize_volume(s['flair'])
    t = norm[norm != 0]
    print(f"📊 Normalization: mean={t.mean():.4f}, std={t.std():.4f}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-06: Case-Level Train/Val Split
# FIX: .copy() before shuffle
# FIX: Check if split JSON already exists
# Split JSON saves to Drive (persistent across sessions)
# ═══════════════════════════════════════════════════════════════

def save_dataset_split(case_ids, train_ratio=0.8):
    random.seed(42)
    shuffled = case_ids.copy()
    random.shuffle(shuffled)
    split_point = int(len(shuffled) * train_ratio)
    split_dict = {'train': shuffled[:split_point], 'val': shuffled[split_point:]}

    output_path = os.path.join(OUTPUT_DIR, 'dataset_split.json')
    try:
        with open(output_path, 'w') as f:
            json.dump(split_dict, f, indent=4)
        print(f"✅ Split saved to: {output_path}")
    except OSError as e:
        print(f"❌ Failed: {e}")
    print(f"   Training: {len(split_dict['train'])}, Validation: {len(split_dict['val'])}")
    return split_dict

split_path = os.path.join(OUTPUT_DIR, 'dataset_split.json')
if os.path.exists(split_path):
    print(f"📄 Loading existing split...")
    with open(split_path) as f:
        dataset_split = json.load(f)
    print(f"   {len(dataset_split['train'])} train / {len(dataset_split['val'])} val")
else:
    dataset_split = save_dataset_split(all_cases)

overlap = set(dataset_split['train']) & set(dataset_split['val'])
print(f"{'❌ LEAKAGE!' if overlap else '✅ No leakage'}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-04: Partition-Aware Slice Extraction — PARALLELIZED
#
# OPTIMIZATION: ProcessPoolExecutor for multi-core processing.
# On A100 High-RAM (8-12 cores): ~15-20 min vs 90+ min sequential.
#
# FIXES FROM PREVIOUS VERSION:
# - Atomic write (.tmp then rename) prevents partial files if killed
# - Error messages captured and returned, not silently swallowed
# - Uses TRAIN_PATH (which may have been updated for nested zips)
# ═══════════════════════════════════════════════════════════════

def process_single_case(args):
    """Load → normalize → extract → save. Runs in a separate process.
    Must be a top-level function (not a closure) for pickling."""
    case_id, data_path, save_dir = args

    try:
        # Load all 5 NIfTI files
        case_path = os.path.join(data_path, case_id)
        data_dict = {}
        for mod in ['t1', 't1ce', 't2', 'flair']:
            fp = os.path.join(case_path, f"{case_id}_{mod}.nii.gz")
            data_dict[mod] = nib.load(fp).get_fdata().astype(np.float32)
        seg_fp = os.path.join(case_path, f"{case_id}_seg.nii.gz")
        data_dict['seg'] = nib.load(seg_fp).get_fdata().astype(np.uint8)

        # Per-volume z-score normalization (brain mask only)
        for mod in ['t1', 't1ce', 't2', 'flair']:
            vol = np.nan_to_num(data_dict[mod], nan=0.0, posinf=0.0, neginf=0.0)
            brain = vol > 0
            if np.any(brain):
                m, s = vol[brain].mean(), vol[brain].std()
                vol = (vol - m) / (s + 1e-8)
                vol[~brain] = 0
            data_dict[mod] = vol.astype(np.float32)

        # Find tumor z-range with margin
        seg = data_dict['seg']
        z_idx = np.any(seg > 0, axis=(0, 1))
        if not np.any(z_idx):
            return case_id, 0, None  # no tumor

        start_z = max(0, np.where(z_idx)[0].min() - 5)
        end_z = min(seg.shape[2], np.where(z_idx)[0].max() + 6)

        # Extract and save each slice with atomic write
        count = 0
        for z in range(start_z, end_z):
            img = np.stack([
                data_dict['t1'][:, :, z], data_dict['t1ce'][:, :, z],
                data_dict['t2'][:, :, z], data_dict['flair'][:, :, z]
            ], axis=0)
            msk = seg[:, :, z]

            # Atomic write: save to .tmp, then rename. If process is
            # killed mid-write, no partial .npz exists to crash loaders.
            final_path = os.path.join(save_dir, f"{case_id}_slice_{count:03d}.npz")
            tmp_path = final_path + '.tmp'
            np.savez_compressed(tmp_path, image=img, mask=msk)
            os.rename(tmp_path, final_path)
            count += 1

        return case_id, count, None
    except Exception as e:
        return case_id, -1, str(e)  # return error message


def extract_partition_parallel(case_ids, partition_name, data_path, num_workers=6):
    """Extract slices using multiple CPU cores in parallel."""
    save_dir = os.path.join(LOCAL_DIR, 'slices', partition_name)
    os.makedirs(save_dir, exist_ok=True)

    args_list = [(cid, data_path, save_dir) for cid in case_ids]
    total_slices = 0
    completed = 0
    errors = []
    start_time = time.time()

    print(f"  [{partition_name}] Processing {len(case_ids)} cases "
          f"with {num_workers} parallel workers...")

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {executor.submit(process_single_case, a): a[0]
                   for a in args_list}

        for future in as_completed(futures):
            case_id, count, err = future.result()
            completed += 1
            if count < 0:
                errors.append((case_id, err))
            else:
                total_slices += count

            if completed % 100 == 0 or completed == len(case_ids):
                elapsed = time.time() - start_time
                rate = completed / elapsed * 60 if elapsed > 0 else 0
                print(f"  [{partition_name}] {completed}/{len(case_ids)} cases, "
                      f"{total_slices} slices, {elapsed/60:.1f}min "
                      f"({rate:.0f} cases/min)")

    elapsed = time.time() - start_time
    print(f"✅ {partition_name}: {total_slices} slices in {elapsed/60:.1f} minutes")
    if errors:
        print(f"⚠️  {len(errors)} cases failed:")
        for cid, err in errors[:5]:
            print(f"    {cid}: {err}")
        if len(errors) > 5:
            print(f"    ... and {len(errors)-5} more")

    return total_slices, errors

print("✅ Extraction functions ready (parallelized, atomic, error-reporting)")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Run Parallel Extraction
#
# SKIP THIS CELL if slices already exist (run 'Load Slices from
# Drive Zip' cell instead).
# ═══════════════════════════════════════════════════════════════

import multiprocessing
num_cores = multiprocessing.cpu_count()
NUM_WORKERS = min(8, num_cores - 1)
print(f"💻 Available CPU cores: {num_cores}, using {NUM_WORKERS} workers")

total_start = time.time()

print("\n📤 Extracting training slices...")
train_count, train_errors = extract_partition_parallel(
    dataset_split['train'], 'train', TRAIN_PATH, num_workers=NUM_WORKERS)

print("\n📤 Extracting validation slices...")
val_count, val_errors = extract_partition_parallel(
    dataset_split['val'], 'val', TRAIN_PATH, num_workers=NUM_WORKERS)

total_elapsed = time.time() - total_start
print(f"\n✅ TOTAL: {train_count} train + {val_count} val slices "
      f"in {total_elapsed/60:.1f} minutes")
print(f"   Errors: {len(train_errors)} train, {len(val_errors)} val")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Transfer Slices to Drive (one large zip file)
# ═══════════════════════════════════════════════════════════════

import shutil

zip_path = '/content/brats_slices'
print('📦 Zipping extracted slices...')
shutil.make_archive(zip_path, 'zip', LOCAL_DIR)
print(f'✅ Created {zip_path}.zip')

drive_zip = os.path.join(OUTPUT_DIR, 'brats_slices.zip')
print(f'📤 Copying to Drive...')
shutil.copy2(f'{zip_path}.zip', drive_zip)
print(f'✅ Saved: {drive_zip} ({os.path.getsize(drive_zip)/1e9:.2f} GB)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Load Slices from Drive Zip (future sessions)
#
# Run THIS cell instead of Cells 3 + 9 + 10 if slices were
# already extracted in a previous session.
# ═══════════════════════════════════════════════════════════════

import zipfile

LOCAL_DIR = '/content/local_slices'
drive_zip = os.path.join(OUTPUT_DIR, 'brats_slices.zip')
local_check = os.path.join(LOCAL_DIR, 'slices', 'train')

if os.path.exists(local_check):
    nt = len([f for f in os.listdir(local_check) if f.endswith('.npz')])
    nv = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','val')) if f.endswith('.npz')])
    print(f'✅ Local slices exist: {nt} train + {nv} val')
elif os.path.exists(drive_zip):
    print(f'📥 Extracting from Drive zip...')
    with zipfile.ZipFile(drive_zip, 'r') as z:
        z.extractall(LOCAL_DIR)
    nt = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','train')) if f.endswith('.npz')])
    nv = len([f for f in os.listdir(
        os.path.join(LOCAL_DIR,'slices','val')) if f.endswith('.npz')])
    print(f'✅ Unzipped: {nt} train + {nv} val')
else:
    print('⚠️  No zip on Drive. Run extraction cells first.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-05: Train-Only Augmentation
#
# FIX: .copy() after np.flip
# FIX: Geometric (image+mask) separated from photometric (image only)
# FIX: borderMode=BORDER_CONSTANT, borderValue=0.0
# FIX: Mask cast float32 before warp, uint8 after
# FIX: Wrapped in Dataset class with augment flag
# ═══════════════════════════════════════════════════════════════

def geometric_augmentation(image, mask):
    """GEOMETRIC: apply to BOTH image and mask. NN interp on mask."""
    if random.random() > 0.5:
        image = np.flip(image, axis=2).copy()  # FIX: .copy()
        mask = np.flip(mask, axis=1).copy()     # FIX: .copy()

    angle = random.uniform(-15, 15)
    h, w = mask.shape
    rot = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)

    aug_img = np.zeros_like(image)
    for c in range(image.shape[0]):
        aug_img[c] = cv2.warpAffine(
            image[c], rot, (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)

    aug_msk = cv2.warpAffine(
        mask.astype(np.float32), rot, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT, borderValue=0.0
    ).astype(np.uint8)

    return aug_img, aug_msk


def photometric_augmentation(image):
    """PHOTOMETRIC: IMAGE ONLY. Never apply to mask."""
    image = image * random.uniform(0.9, 1.1)
    if random.random() > 0.5:
        noise = np.random.normal(0, 0.02, image.shape).astype(np.float32)
        image = image + noise
    return image


def apply_augmentation(image, mask):
    """Geometric first (both), then photometric (image only)."""
    image, mask = geometric_augmentation(image, mask)
    image = photometric_augmentation(image)
    return image, mask


class BraTSSliceDataset:
    """Dataset wrapper. augment=True for train, False for val."""
    def __init__(self, slice_dir, augment=False):
        self.slice_dir = slice_dir
        self.augment = augment
        self.file_list = sorted(
            [f for f in os.listdir(slice_dir) if f.endswith('.npz')])
        print(f"📦 {len(self.file_list)} slices "
              f"(augment={'ON' if augment else 'OFF'})")

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.slice_dir, self.file_list[idx]))
        image = data['image'].astype(np.float32)
        mask = data['mask'].astype(np.uint8)

        if self.augment:
            image, mask = apply_augmentation(image, mask)

        return image.astype(np.float32), mask.astype(np.int64)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Create Datasets and Verify (reads from local SSD)
# ═══════════════════════════════════════════════════════════════

train_dir = os.path.join(LOCAL_DIR, 'slices', 'train')
val_dir = os.path.join(LOCAL_DIR, 'slices', 'val')

train_dataset = BraTSSliceDataset(train_dir, augment=True)
val_dataset = BraTSSliceDataset(val_dir, augment=False)

if len(train_dataset) > 0:
    img_t, msk_t = train_dataset[0]
    print(f'\n🔬 Train: {img_t.shape} {img_t.dtype}, mask labels: {np.unique(msk_t)}')
if len(val_dataset) > 0:
    img_v, msk_v = val_dataset[0]
    print(f'🔬 Val:   {img_v.shape} {img_v.dtype}, mask labels: {np.unique(msk_v)}')
print('\n🚀 Phase 1 complete.')


---
# Phase 2: SEG-07 through SEG-18 + EVAL-01/02
## U-Net Segmentation + Ablation Experiments (TensorFlow/Keras)

**Prerequisites:** Phase 1 cells above must have run successfully. Slices must exist at `LOCAL_DIR/slices/`.

### What this phase covers
- **EVAL-01**: Dataset statistics (class balance, tumor distribution)
- **EVAL-02**: 4-modality side-by-side visualizations + mask overlay
- **SEG-07**: U-Net architecture — encoder-decoder with skip connections
- **SEG-08**: 4-channel early fusion input layer
- **SEG-09**: Soft Dice + BCE combined loss, IoU/Dice metrics
- **SEG-10**: Training loop (Adam + ReduceOnPlateau)
- **SEG-11**: Train baseline U-Net (HANDOFF task)
- **SEG-12**: Per-class evaluation (Dice/IoU for WT/TC/ET)
- **SEG-13**: Segmentation overlay visualizations
- **SEG-15**: Drop-T1ce ablation (Priority 1)
- **SEG-17**: Drop-FLAIR ablation (Priority 2)
- **SEG-14**: Drop-T1 ablation (post-presentation)
- **SEG-16**: Drop-T2 ablation (post-presentation)
- **SEG-18**: Compile ablation table (HANDOFF task)

### Architecture decisions (per Dr. Lai's lectures)
- **Week 4**: U-Net with 2 conv-BN-ReLU blocks per encoder level, skip connections via feature map concatenation
- **Week 5**: Dice+BCE for training loss (avoids zero-overlap gradient problem), IoU as primary evaluation metric
- **Week 6**: Keras Functional API
- **No pretrained weights**: "I always retrain everything from scratch" (Week 4)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# TensorFlow/Keras Imports + GPU Memory Growth
# ═══════════════════════════════════════════════════════════════

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import glob

# Grow GPU memory instead of grabbing all at once
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU: {gpus[0].name}')
else:
    print('⚠️  No GPU detected. Training will be slow.')

# Slice directories (local SSD) + model dir (Drive)
SLICE_DIR_TRAIN = os.path.join(LOCAL_DIR, 'slices', 'train')
SLICE_DIR_VAL = os.path.join(LOCAL_DIR, 'slices', 'val')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Train slices: {len(glob.glob(os.path.join(SLICE_DIR_TRAIN, "*.npz")))}')
print(f'Val slices:   {len(glob.glob(os.path.join(SLICE_DIR_VAL, "*.npz")))}')
print(f'TensorFlow:   {tf.__version__}')
print(f'Model dir:    {MODEL_DIR}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EVAL-01: Dataset Statistics
# Class balance + tumor pixel distribution
# ═══════════════════════════════════════════════════════════════

def compute_dataset_stats(slice_dir, max_samples=500):
    files = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))[:max_samples]
    if not files:
        print(f'No .npz in {slice_dir}')
        return None

    label_counts = {0: 0, 1: 0, 2: 0, 4: 0}
    tumor_pixels = []
    for f in files:
        msk = np.load(f)['mask']
        for label in [0, 1, 2, 4]:
            label_counts[label] += int(np.sum(msk == label))
        tumor_pixels.append(int(np.sum(msk > 0)))

    total = sum(label_counts.values())
    names = {0: 'Background', 1: 'Necrotic Core', 2: 'Edema', 4: 'Enhancing Tumor'}
    print(f'📊 EVAL-01: Dataset Statistics ({len(files)} slices sampled)\n')
    for label, count in label_counts.items():
        pct = 100 * count / total if total > 0 else 0
        print(f'  {names[label]:20s} (label {label}): {count:>12,} px ({pct:.2f}%)')
    tp = np.array(tumor_pixels)
    print(f'\n  Tumor px/slice — Mean: {tp.mean():.0f}, '
          f'Median: {np.median(tp):.0f}, Min: {tp.min()}, Max: {tp.max()}')
    return label_counts

train_stats = compute_dataset_stats(SLICE_DIR_TRAIN)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# EVAL-02: Sample 4-Modality Visualizations + Mask Overlay
# ═══════════════════════════════════════════════════════════════

def visualize_sample(slice_dir, sample_idx=0):
    files = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))
    if sample_idx >= len(files):
        return

    data = np.load(files[sample_idx])
    img, msk = data['image'], data['mask']  # (4,240,240), (240,240)
    fname = os.path.basename(files[sample_idx])

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.suptitle(fname, fontsize=12)
    for i, (name, ax) in enumerate(zip(['T1', 'T1ce', 'T2', 'FLAIR'], axes[:4])):
        ax.imshow(img[i], cmap='gray')
        ax.set_title(name); ax.axis('off')

    # Colour overlay: red=necrotic, green=edema, yellow=enhancing
    mask_rgb = np.zeros((*msk.shape, 3), dtype=np.float32)
    mask_rgb[msk == 1] = [1, 0, 0]
    mask_rgb[msk == 2] = [0, 1, 0]
    mask_rgb[msk == 4] = [1, 1, 0]
    axes[4].imshow(img[1], cmap='gray')
    axes[4].imshow(mask_rgb, alpha=0.4)
    axes[4].set_title('Mask Overlay'); axes[4].axis('off')
    plt.tight_layout(); plt.show()

train_files = sorted(glob.glob(os.path.join(SLICE_DIR_TRAIN, '*.npz')))
for idx in random.sample(range(len(train_files)), min(3, len(train_files))):
    visualize_sample(SLICE_DIR_TRAIN, idx)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-07: U-Net Architecture + SEG-08: Configurable Early Fusion
#
# Dr. Lai Week 4:
#   'Each encoder usually has two convolution blocks...
#    convolution, batch normalization, and ReLU. Repeat twice.'
#   'U-Net is using the skip connection, passing the feature map.'
#   'They are concatenated together.'
#
# Input shape configurable for ablations:
#   4 channels = all modalities (baseline)
#   3 channels = ablation (one modality dropped)
#
# Output: 3 sigmoid channels (WT, TC, ET — hierarchical, not
# mutually exclusive, so sigmoid not softmax).
# ═══════════════════════════════════════════════════════════════

def conv_block(x, filters, name_prefix):
    """Two conv-BN-ReLU blocks (Dr. Lai Week 4 convention)."""
    x = layers.Conv2D(filters, 3, padding='same', name=f'{name_prefix}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.ReLU(name=f'{name_prefix}_relu1')(x)
    x = layers.Conv2D(filters, 3, padding='same', name=f'{name_prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    x = layers.ReLU(name=f'{name_prefix}_relu2')(x)
    return x

def encoder_block(x, filters, name_prefix):
    """Conv block + pool. Returns pre-pool FM (skip) and pooled."""
    ppf = conv_block(x, filters, name_prefix)
    pooled = layers.MaxPooling2D(2, name=f'{name_prefix}_pool')(ppf)
    pooled = layers.Dropout(0.1, name=f'{name_prefix}_drop')(pooled)
    return ppf, pooled

def decoder_block(x, skip, filters, name_prefix):
    """Transpose conv + concat skip + conv block."""
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same',
                                name=f'{name_prefix}_upconv')(x)
    x = layers.Concatenate(name=f'{name_prefix}_concat')([x, skip])
    x = conv_block(x, filters, name_prefix)
    return x

def build_unet(input_channels=4, num_classes=3):
    """2D U-Net with configurable input channels for ablations.
    Encoder: 64→128→256→512, Bottleneck: 1024, Decoder: mirror."""
    inputs = keras.Input(shape=(240, 240, input_channels), name='input')
    s1, p1 = encoder_block(inputs, 64,  'enc1')   # 240→120
    s2, p2 = encoder_block(p1,     128, 'enc2')   # 120→60
    s3, p3 = encoder_block(p2,     256, 'enc3')   # 60→30
    s4, p4 = encoder_block(p3,     512, 'enc4')   # 30→15
    b = conv_block(p4, 1024, 'bottleneck')
    d4 = decoder_block(b,  s4, 512, 'dec4')
    d3 = decoder_block(d4, s3, 256, 'dec3')
    d2 = decoder_block(d3, s2, 128, 'dec2')
    d1 = decoder_block(d2, s1, 64,  'dec1')
    outputs = layers.Conv2D(num_classes, 1, activation='sigmoid',
                             name='output')(d1)
    return keras.Model(inputs, outputs, name=f'UNet_{input_channels}ch')

# Baseline 4-channel model
model = build_unet(input_channels=4)
model.summary()

dummy = np.random.rand(2, 240, 240, 4).astype(np.float32)
out = model(dummy, training=False)
print(f'\n✅ Input:  {dummy.shape}')
print(f'✅ Output: {out.shape}  (expected: (2, 240, 240, 3))')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-09: Soft Dice + BCE Loss + IoU/Dice Metrics
#
# Dr. Lai Week 5:
#   'I personally prefer IOU... dice usually will give more
#    optimistic output... inflated'
#   'Using dice as a training... using IOU for the validation'
# ═══════════════════════════════════════════════════════════════

def soft_dice_loss(y_true, y_pred, smooth=1e-6):
    inter = tf.reduce_sum(y_true * y_pred, axis=(1, 2))
    denom = tf.reduce_sum(y_true, axis=(1, 2)) + tf.reduce_sum(y_pred, axis=(1, 2))
    return 1.0 - tf.reduce_mean((2.0 * inter + smooth) / (denom + smooth))

def dice_bce_loss(y_true, y_pred):
    return soft_dice_loss(y_true, y_pred) + \
           tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred))

def iou_metric(y_true, y_pred):
    yp = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * yp, axis=(1, 2))
    union = tf.reduce_sum(y_true, axis=(1, 2)) + tf.reduce_sum(yp, axis=(1, 2)) - inter
    return tf.reduce_mean((inter + 1e-6) / (union + 1e-6))

def dice_metric(y_true, y_pred):
    yp = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * yp, axis=(1, 2))
    denom = tf.reduce_sum(y_true, axis=(1, 2)) + tf.reduce_sum(yp, axis=(1, 2))
    return tf.reduce_mean((2.0 * inter + 1e-6) / (denom + 1e-6))

dt = np.random.randint(0, 2, (2, 240, 240, 3)).astype(np.float32)
dp = np.random.rand(2, 240, 240, 3).astype(np.float32)
print(f'Dice+BCE loss: {dice_bce_loss(dt, dp):.4f}')
print(f'IoU metric:    {iou_metric(dt, dp):.4f}')
print(f'Dice metric:   {dice_metric(dt, dp):.4f}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Data Loader: .npz slices → tf.data.Dataset
#
# Conversions:
#   channels-first (4, 240, 240) → channels-last (240, 240, 4)
#   BraTS labels (0,1,2,4) → 3 binary channels (WT, TC, ET)
#
# Ablation support via modality_indices:
#   None         = all 4 modalities (baseline)
#   [0, 2, 3]    = drop T1ce
#   [0, 1, 2]    = drop FLAIR
#   [1, 2, 3]    = drop T1
#   [0, 1, 3]    = drop T2
# ═══════════════════════════════════════════════════════════════

MODALITY_NAMES = ['T1', 'T1ce', 'T2', 'FLAIR']

def brats_labels_to_channels(mask):
    """Integer labels → 3 binary channels (WT, TC, ET)."""
    wt = np.float32(mask > 0)
    tc = np.float32((mask == 1) | (mask == 4))
    et = np.float32(mask == 4)
    return np.stack([wt, tc, et], axis=-1)

def load_npz_for_tf(fpath, modality_indices=None):
    data = np.load(fpath)
    img = data['image'].astype(np.float32)
    msk = data['mask'].astype(np.uint8)
    if modality_indices is not None:
        img = img[modality_indices]
    img = np.transpose(img, (1, 2, 0))       # → channels-last
    msk = brats_labels_to_channels(msk)      # → (240, 240, 3)
    return img, msk

def create_tf_dataset(slice_dir, batch_size=8, modality_indices=None, shuffle=True):
    file_list = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))
    n_ch = 4 if modality_indices is None else len(modality_indices)
    label = 'all' if modality_indices is None else '+'.join([MODALITY_NAMES[i] for i in modality_indices])
    print(f'📦 {len(file_list)} slices from {os.path.basename(slice_dir)} ({label}, {n_ch}ch)')

    def gen():
        fs = file_list.copy()
        if shuffle:
            np.random.shuffle(fs)
        for f in fs:
            yield load_npz_for_tf(f, modality_indices=modality_indices)

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((240, 240, n_ch), tf.float32),
        tf.TensorSpec((240, 240, 3), tf.float32)))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE), len(file_list)

# Baseline datasets
BATCH_SIZE = 8
train_ds, train_count = create_tf_dataset(SLICE_DIR_TRAIN, BATCH_SIZE, shuffle=True)
val_ds, val_count = create_tf_dataset(SLICE_DIR_VAL, BATCH_SIZE, shuffle=False)

for ib, mb in train_ds.take(1):
    print(f'\n✅ Batch: images {ib.shape}, masks {mb.shape}')
    print(f'   Range: [{ib.numpy().min():.2f}, {ib.numpy().max():.2f}]')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-10 + SEG-11: Train Baseline U-Net (HANDOFF TASK)
#
# Dr. Lai Week 4: 'I always retrain everything from scratch' —
# no pretrained ImageNet weights for medical imaging.
# ═══════════════════════════════════════════════════════════════

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=dice_bce_loss,
    metrics=[dice_metric, iou_metric]
)

callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(MODEL_DIR, 'unet_baseline_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=1),
]

EPOCHS = 50
history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1
)

print(f'\n✅ SEG-11: Baseline training complete.')
print(f'   Best val loss: {min(history.history["val_loss"]):.4f}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Training History Visualization
# ═══════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, name in zip(axes,
                          ['loss', 'dice_metric', 'iou_metric'],
                          ['Dice+BCE Loss', 'Dice Score', 'IoU Score']):
    ax.plot(history.history[key], label='Train')
    ax.plot(history.history[f'val_{key}'], label='Val')
    ax.set_title(name); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves_baseline.png'), dpi=150)
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-12: Per-Class Evaluation (reusable for ablations)
# ═══════════════════════════════════════════════════════════════

def evaluate_per_class(model, dataset, class_names=['WT', 'TC', 'ET'], label=''):
    all_dice = {c: [] for c in class_names}
    all_iou = {c: [] for c in class_names}

    for ib, mb in dataset:
        pb = tf.cast(model(ib, training=False) > 0.5, tf.float32)
        for ci, cn in enumerate(class_names):
            yt, yp = mb[:, :, :, ci], pb[:, :, :, ci]
            inter = tf.reduce_sum(yt * yp, axis=(1, 2))
            union = tf.reduce_sum(yt, axis=(1, 2)) + tf.reduce_sum(yp, axis=(1, 2)) - inter
            denom = tf.reduce_sum(yt, axis=(1, 2)) + tf.reduce_sum(yp, axis=(1, 2))
            all_iou[cn].extend(((inter + 1e-6) / (union + 1e-6)).numpy().tolist())
            all_dice[cn].extend(((2 * inter + 1e-6) / (denom + 1e-6)).numpy().tolist())

    print(f'📊 Evaluation: {label}' if label else '📊 Evaluation')
    print(f'{"Region":<8} {"Dice":>10} {"± std":>10} {"IoU":>10} {"± std":>10}')
    print('-' * 52)
    results = {}
    for cn in class_names:
        d, i = np.array(all_dice[cn]), np.array(all_iou[cn])
        print(f'{cn:<8} {d.mean():>10.4f} {d.std():>10.4f} {i.mean():>10.4f} {i.std():>10.4f}')
        results[cn] = {
            'dice_mean': float(d.mean()), 'dice_std': float(d.std()),
            'iou_mean': float(i.mean()), 'iou_std': float(i.std())
        }
    results['Mean'] = {
        'dice_mean': float(np.mean([results[c]['dice_mean'] for c in class_names])),
        'iou_mean': float(np.mean([results[c]['iou_mean'] for c in class_names])),
    }
    print('-' * 52)
    print(f'{"Mean":<8} {results["Mean"]["dice_mean"]:>10.4f} {"":>10} '
          f'{results["Mean"]["iou_mean"]:>10.4f}')
    return results

baseline_results = evaluate_per_class(model, val_ds, label='Baseline (all 4 modalities)')

with open(os.path.join(OUTPUT_DIR, 'baseline_metrics.json'), 'w') as f:
    json.dump(baseline_results, f, indent=2)
print('\n✅ Saved baseline_metrics.json')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-13: Segmentation Overlay Visualizations
# FLAIR | Ground Truth | Prediction  (WT=green, TC=yellow, ET=red)
# ═══════════════════════════════════════════════════════════════

def create_overlay(base, mask_3ch, alpha=0.4):
    b = (base - base.min()) / (base.max() - base.min() + 1e-8)
    rgb = np.stack([b, b, b], axis=-1)
    overlay = np.zeros_like(rgb)
    overlay[mask_3ch[:, :, 0] > 0.5] = [0, 1, 0]
    overlay[mask_3ch[:, :, 1] > 0.5] = [1, 1, 0]
    overlay[mask_3ch[:, :, 2] > 0.5] = [1, 0, 0]
    wt = mask_3ch[:, :, 0:1] > 0.5
    return np.clip(np.where(wt, rgb * (1 - alpha) + overlay * alpha, rgb), 0, 1)

def show_overlays(model, dataset, n=5, save_name='overlays.png'):
    samples = []
    for ib, mb in dataset:
        pb = model(ib, training=False)
        for i in range(ib.shape[0]):
            if tf.reduce_sum(mb[i]).numpy() > 100:
                samples.append((ib[i].numpy(), mb[i].numpy(), pb[i].numpy()))
            if len(samples) >= n:
                break
        if len(samples) >= n:
            break

    fig, axes = plt.subplots(len(samples), 3, figsize=(15, 5 * len(samples)))
    if len(samples) == 1:
        axes = axes[np.newaxis, :]

    for row, (img, gt, pred) in enumerate(samples):
        flair = img[:, :, -1]  # last channel (FLAIR for baseline, or last-available for ablations)
        axes[row, 0].imshow(flair, cmap='gray')
        axes[row, 0].set_title('FLAIR'); axes[row, 0].axis('off')
        axes[row, 1].imshow(create_overlay(flair, gt))
        axes[row, 1].set_title('Ground Truth'); axes[row, 1].axis('off')
        axes[row, 2].imshow(create_overlay(flair, (pred > 0.5).astype(np.float32)))
        axes[row, 2].set_title('Prediction'); axes[row, 2].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()
    print(f'✅ Saved: {save_name}')

show_overlays(model, val_ds, save_name='segmentation_overlays_baseline.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-15: Drop-T1ce Ablation (PRIORITY 1)
#
# T1ce (gadolinium-enhanced T1) is most critical for ET detection.
# Clinically relevant: T1ce is sometimes contraindicated (kidney
# patients). Expected: ET Dice degrades significantly.
# ═══════════════════════════════════════════════════════════════

DROP_T1CE = [0, 2, 3]  # keep T1, T2, FLAIR

train_ds_no_t1ce, _ = create_tf_dataset(SLICE_DIR_TRAIN, BATCH_SIZE, DROP_T1CE, shuffle=True)
val_ds_no_t1ce, _ = create_tf_dataset(SLICE_DIR_VAL, BATCH_SIZE, DROP_T1CE, shuffle=False)

model_no_t1ce = build_unet(input_channels=3)
model_no_t1ce.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=dice_bce_loss, metrics=[dice_metric, iou_metric])

history_no_t1ce = model_no_t1ce.fit(
    train_ds_no_t1ce, validation_data=val_ds_no_t1ce,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, 'unet_drop_t1ce.keras'),
            monitor='val_loss', save_best_only=True),
    ], verbose=1)

drop_t1ce_results = evaluate_per_class(model_no_t1ce, val_ds_no_t1ce,
                                        label='Drop-T1ce (T1+T2+FLAIR)')
show_overlays(model_no_t1ce, val_ds_no_t1ce, n=3,
              save_name='segmentation_overlays_drop_t1ce.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-17: Drop-FLAIR Ablation (PRIORITY 2)
# FLAIR is critical for WT (edema). Expected: WT Dice degrades.
# ═══════════════════════════════════════════════════════════════

DROP_FLAIR = [0, 1, 2]  # keep T1, T1ce, T2

train_ds_no_flair, _ = create_tf_dataset(SLICE_DIR_TRAIN, BATCH_SIZE, DROP_FLAIR, shuffle=True)
val_ds_no_flair, _ = create_tf_dataset(SLICE_DIR_VAL, BATCH_SIZE, DROP_FLAIR, shuffle=False)

model_no_flair = build_unet(input_channels=3)
model_no_flair.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=dice_bce_loss, metrics=[dice_metric, iou_metric])

history_no_flair = model_no_flair.fit(
    train_ds_no_flair, validation_data=val_ds_no_flair,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, 'unet_drop_flair.keras'),
            monitor='val_loss', save_best_only=True),
    ], verbose=1)

drop_flair_results = evaluate_per_class(model_no_flair, val_ds_no_flair,
                                         label='Drop-FLAIR (T1+T1ce+T2)')
show_overlays(model_no_flair, val_ds_no_flair, n=3,
              save_name='segmentation_overlays_drop_flair.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-14: Drop-T1 Ablation (post-presentation if time)
# ═══════════════════════════════════════════════════════════════

DROP_T1 = [1, 2, 3]  # keep T1ce, T2, FLAIR

train_ds_no_t1, _ = create_tf_dataset(SLICE_DIR_TRAIN, BATCH_SIZE, DROP_T1, shuffle=True)
val_ds_no_t1, _ = create_tf_dataset(SLICE_DIR_VAL, BATCH_SIZE, DROP_T1, shuffle=False)

model_no_t1 = build_unet(input_channels=3)
model_no_t1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=dice_bce_loss, metrics=[dice_metric, iou_metric])

history_no_t1 = model_no_t1.fit(
    train_ds_no_t1, validation_data=val_ds_no_t1,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, 'unet_drop_t1.keras'),
            monitor='val_loss', save_best_only=True),
    ], verbose=1)

drop_t1_results = evaluate_per_class(model_no_t1, val_ds_no_t1,
                                      label='Drop-T1 (T1ce+T2+FLAIR)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-16: Drop-T2 Ablation (post-presentation if time)
# ═══════════════════════════════════════════════════════════════

DROP_T2 = [0, 1, 3]  # keep T1, T1ce, FLAIR

train_ds_no_t2, _ = create_tf_dataset(SLICE_DIR_TRAIN, BATCH_SIZE, DROP_T2, shuffle=True)
val_ds_no_t2, _ = create_tf_dataset(SLICE_DIR_VAL, BATCH_SIZE, DROP_T2, shuffle=False)

model_no_t2 = build_unet(input_channels=3)
model_no_t2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=dice_bce_loss, metrics=[dice_metric, iou_metric])

history_no_t2 = model_no_t2.fit(
    train_ds_no_t2, validation_data=val_ds_no_t2,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, 'unet_drop_t2.keras'),
            monitor='val_loss', save_best_only=True),
    ], verbose=1)

drop_t2_results = evaluate_per_class(model_no_t2, val_ds_no_t2,
                                      label='Drop-T2 (T1+T1ce+FLAIR)')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-18: Compile Ablation Table (HANDOFF TASK)
#
# Produces: ablation_results.csv, ablation_results.json,
# ablation_chart.png — deliverables for Sipho (INT-02 recovery)
# and Manoj (EVAL-06 comparison tables).
# ═══════════════════════════════════════════════════════════════

import pandas as pd

all_results = {'Baseline (all 4)': baseline_results}
for name, varname in [('Drop-T1ce', 'drop_t1ce_results'),
                      ('Drop-FLAIR', 'drop_flair_results'),
                      ('Drop-T1', 'drop_t1_results'),
                      ('Drop-T2', 'drop_t2_results')]:
    try:
        all_results[name] = eval(varname)
    except NameError:
        pass  # ablation not run yet

# Flatten into dataframe
rows = []
for config_name, r in all_results.items():
    for region in ['WT', 'TC', 'ET', 'Mean']:
        if region in r:
            rows.append({
                'Configuration': config_name,
                'Region': region,
                'Dice': round(r[region]['dice_mean'], 4),
                'IoU': round(r[region].get('iou_mean', float('nan')), 4),
            })
df = pd.DataFrame(rows)
print('📋 SEG-18: Ablation Results')
print(df.to_string(index=False))

# Save CSV and JSON
csv_path = os.path.join(OUTPUT_DIR, 'ablation_results.csv')
df.to_csv(csv_path, index=False)
json_path = os.path.join(OUTPUT_DIR, 'ablation_results.json')
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\n✅ Saved: {csv_path}')
print(f'✅ Saved: {json_path}')

# Bar chart
pivot = df[df['Region'].isin(['WT', 'TC', 'ET'])].pivot(
    index='Configuration', columns='Region', values='Dice')
fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind='bar', ax=ax, color=['#2ca02c', '#ffbb33', '#d62728'])
ax.set_title('Ablation Study: Dice Score by Tumor Region', fontsize=14)
ax.set_ylabel('Dice Score'); ax.set_xlabel('Modality Configuration')
ax.legend(title='Region', loc='lower left')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, 'ablation_chart.png')
plt.savefig(chart_path, dpi=150)
plt.show()
print(f'✅ Saved: {chart_path}')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# SEG-11 + SEG-18 Handoff Checklists
# ═══════════════════════════════════════════════════════════════

print('═' * 60)
print('SEG-11 HANDOFF CHECKLIST (Baseline U-Net)')
print('═' * 60)
print(f'1. Checkpoint: {os.path.join(MODEL_DIR, "unet_baseline_best.keras")}')
print(f'2. Input:  (batch, 240, 240, 4) channels-last float32')
print(f'   Output: (batch, 240, 240, 3) sigmoid [0,1]')
print(f'   Input channels: [T1, T1ce, T2, FLAIR]')
print(f'   Output channels: [WT, TC, ET] (hierarchical binary)')
print(f'3. Dtype: float32')
print(f'4. Normalization: z-scored per-volume (Yue preprocessing)')
print(f'5. Inference:')
print(f'   model = keras.models.load_model(path, custom_objects={{')
print(f'       "dice_bce_loss": dice_bce_loss,')
print(f'       "dice_metric": dice_metric,')
print(f'       "iou_metric": iou_metric}})')
print(f'   pred = model(input_tensor, training=False)')
print(f'   mask = (pred > 0.5).numpy()')
print(f'6. Sanity check: segmentation_overlays_baseline.png')
print(f'7. Framework: TensorFlow/Keras, channels-last')
print()
print('═' * 60)
print('SEG-18 HANDOFF CHECKLIST (Ablation Table)')
print('═' * 60)
print(f'1. CSV:   {os.path.join(OUTPUT_DIR, "ablation_results.csv")}')
print(f'2. JSON:  {os.path.join(OUTPUT_DIR, "ablation_results.json")}')
print(f'3. Chart: {os.path.join(OUTPUT_DIR, "ablation_chart.png")}')
print(f'4. Configurations: {list(all_results.keys())}')
print(f'5. Regions: WT, TC, ET, Mean')
print(f'6. Metrics: Dice mean/std, IoU mean/std')
print(f'7. Per-ablation overlays: segmentation_overlays_drop_*.png')
print('═' * 60)
print('\n🚀 Phase 2 complete.')


In [ ]:
import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Tuple

# FIX #3 (CRITICAL): Use channels-first globally to match Yue's preprocessing
# output format (4, 240, 240). Eliminates the need for to_channels_last transposes.
tf.keras.backend.set_image_data_format('channels_first')

print("TensorFlow version:", tf.__version__)
print("Image data format :", tf.keras.backend.image_data_format())  # should print 'channels_first'

---
# Phase 3: Conditional Multimodal VAE (Pattie's work, integrated)
## VAE-01 through VAE-05 — Modality Synthesis

**Owner:** Pattie. **Integrated by:** David.

**What this does:** Synthesizes a missing MRI modality from the available ones.
This addresses the clinical scenario where one modality (most critically T1ce,
which requires gadolinium contrast and is contraindicated for kidney patients)
is unavailable. The synthesized modality is fed back into the U-Net for
tumor segmentation (INT-01 integration).

### Three synthesis tasks
| Task | Inputs | Output | Jira |
|------|--------|--------|------|
| **A** | T1 + T2 + FLAIR | T1ce | VAE-04a (presentation-critical) |
| **B** | T1 + T1ce + T2 | FLAIR | VAE-04b (stretch) |
| **C** | T1ce + FLAIR | T1 + T2 | VAE-04c (final submission) |

### Integration notes (what I changed when merging Pattie's notebook)
1. **Removed** duplicate `drive.mount()` and `DATA_DIR` setup — reuses Phase 1 paths.
2. **Removed** duplicate `.npz` discovery — reuses `SLICE_DIR_TRAIN/VAL` from Phase 1.
3. **FIXED CRITICAL CHANNEL ORDER BUG**: Pattie's code assumed channel order
   `[T1, T2, T1ce, FLAIR]` but SEG-04 actually saves as `[T1, T1ce, T2, FLAIR]`.
   The `MODALITY_CHANNEL` mapping below corrects this.
4. **Replaced** 2-slice demo training with a proper `tf.data.Dataset` pipeline
   and per-epoch iteration over the full training set.
5. **Added** validation split evaluation with SSIM and PSNR metrics (for EVAL-03).
6. **Added** checkpoint save to Drive using Phase 1's `MODEL_DIR`.

### Architecture (unchanged from Pattie's code)
- **Encoder**: 4 stride-2 conv blocks (32→64→128→256) + GAP + Dense(512) + Dropout
- **Latent**: μ / log σ² heads, dim 128, reparameterization sampling
- **Decoder**: Dense(512) → Dense(256·15·15) → Reshape → 4 transposed convs (256→32) → 1×1 Conv
- **Output**: Linear activation (z-scored data, NOT sigmoid)
- **Loss**: MSE + β·KL with β-annealing over 10 epochs


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE Configuration + Critical Channel-Order Correction
# ═══════════════════════════════════════════════════════════════

from dataclasses import dataclass
from typing import Tuple
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import glob

# CRITICAL: SEG-04 saves channels as [T1, T1ce, T2, FLAIR]
# Pattie's notebook assumed [T1, T2, T1ce, FLAIR] — WRONG for our data.
MODALITY_CHANNEL = {
    'T1':    0,
    'T1ce':  1,  # <-- this was mistakenly index 2 in Pattie's code
    'T2':    2,  # <-- this was mistakenly index 1 in Pattie's code
    'FLAIR': 3,
}

# VAE needs channels-first to keep Pattie's architecture intact.
# TensorFlow has a global setting for this. The U-Net in Phase 2
# uses channels-last, but Keras handles both — we set channels-first
# here for the VAE functions, and the U-Net's explicit shape specs
# still work correctly.
tf.keras.backend.set_image_data_format('channels_first')
print(f"Image data format: {tf.keras.backend.image_data_format()}")

@dataclass
class VAEConfig:
    image_size: Tuple[int, int] = (240, 240)
    in_channels: int = 3
    out_channels: int = 1
    latent_dim: int = 128
    beta: float = 1.0
    learning_rate: float = 1e-4
    batch_size: int = 8
    encoder_filters: Tuple[int, ...] = (32, 64, 128, 256)
    decoder_filters: Tuple[int, ...] = (256, 128, 64, 32)
    dropout_rate: float = 0.2
    seed: int = 42

class BetaScheduler:
    """Linearly anneal beta from 0 to beta_max over warmup_epochs.
    Prevents KL collapse by letting the encoder learn features first."""
    def __init__(self, beta_max: float = 1.0, warmup_epochs: int = 10):
        self.beta_max = beta_max
        self.warmup_epochs = warmup_epochs
    def get(self, epoch: int) -> float:
        if self.warmup_epochs == 0:
            return self.beta_max
        return float(min(self.beta_max, self.beta_max * epoch / self.warmup_epochs))

print('✅ VAE config + channel correction loaded')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-01: Encoder + Sampling + Decoder Architecture
# (Pattie's code, channels-first, linear output, GAP bottleneck)
# ═══════════════════════════════════════════════════════════════

class Sampling(layers.Layer):
    """Reparameterization: z = mu + sigma * epsilon."""
    def call(self, inputs, training=None):
        mu, logvar = inputs
        eps = tf.random.normal(shape=tf.shape(mu))
        sigma = tf.exp(0.5 * logvar)
        return mu + sigma * eps

def build_encoder(config: VAEConfig):
    h, w = config.image_size
    x_in = keras.Input(shape=(config.in_channels, h, w), name='encoder_input')
    x = x_in
    for i, f in enumerate(config.encoder_filters):
        x = layers.Conv2D(f, 3, strides=2, padding='same',
                           data_format='channels_first',
                           name=f'enc_conv_{i}')(x)
        x = layers.BatchNormalization(axis=1, name=f'enc_bn_{i}')(x)
        x = layers.LeakyReLU(negative_slope=0.2, name=f'enc_lrelu_{i}')(x)
    # 240 → 120 → 60 → 30 → 15
    x = layers.GlobalAveragePooling2D(data_format='channels_first',
                                        name='enc_gap')(x)
    x = layers.Dense(512, activation='relu', name='enc_dense')(x)
    x = layers.Dropout(config.dropout_rate, name='enc_dropout')(x)
    mu = layers.Dense(config.latent_dim, name='z_mu')(x)
    logvar = layers.Dense(config.latent_dim, name='z_logvar')(x)
    z = Sampling(name='z_sampling')([mu, logvar])
    return keras.Model(x_in, [mu, logvar, z], name='encoder')

def build_decoder(config: VAEConfig):
    """Linear output activation — required for z-scored data."""
    z_in = keras.Input(shape=(config.latent_dim,), name='z_input')
    x = layers.Dense(512, activation='relu', name='dec_dense_0')(z_in)
    x = layers.Dropout(config.dropout_rate, name='dec_dropout')(x)
    x = layers.Dense(256 * 15 * 15, activation='relu', name='dec_dense_1')(x)
    x = layers.Reshape((256, 15, 15), name='dec_reshape')(x)
    for i, f in enumerate(config.decoder_filters):
        x = layers.Conv2DTranspose(f, 3, strides=2, padding='same',
                                    data_format='channels_first',
                                    name=f'dec_deconv_{i}')(x)
        x = layers.BatchNormalization(axis=1, name=f'dec_bn_{i}')(x)
        x = layers.LeakyReLU(negative_slope=0.2, name=f'dec_lrelu_{i}')(x)
    # Linear output — z-scored data spans ~[-3,+4], sigmoid would destroy it
    x = layers.Conv2D(config.out_channels, 3, padding='same',
                       activation=None, data_format='channels_first',
                       name='decoder_output_conv')(x)
    return keras.Model(z_in, x, name='decoder')

class ConditionalVAE(keras.Model):
    def __init__(self, config: VAEConfig, **kwargs):
        super().__init__(**kwargs)
        self.config = config
        self.encoder = build_encoder(config)
        self.decoder = build_decoder(config)
        self.total_loss_tracker = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker = keras.metrics.Mean(name='recon_loss')
        self.kl_loss_tracker = keras.metrics.Mean(name='kl_loss')

    def call(self, x, training=False):
        mu, logvar, z = self.encoder(x, training=training)
        return self.decoder(z, training=training), mu, logvar, z

    def compute_losses(self, x, y, training=False, beta=None):
        if beta is None: beta = self.config.beta
        recon, mu, logvar, z = self(x, training=training)
        recon_loss = tf.reduce_mean(
            tf.reduce_mean(tf.math.squared_difference(y, recon), axis=(1,2,3))
        )
        kl_loss = tf.reduce_mean(
            -0.5 * tf.reduce_sum(1.0 + logvar - tf.square(mu) - tf.exp(logvar), axis=1)
        )
        return recon_loss + beta * kl_loss, recon_loss, kl_loss, recon, mu, logvar, z

    def train_step_batch(self, optimizer, x, y, beta=None):
        if beta is None: beta = self.config.beta
        with tf.GradientTape() as tape:
            total, recon_l, kl, recon, mu, logvar, z = self.compute_losses(
                x, y, training=True, beta=beta)
        grads = tape.gradient(total, self.trainable_variables)
        optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.total_loss_tracker.update_state(total)
        self.recon_loss_tracker.update_state(recon_l)
        self.kl_loss_tracker.update_state(kl)
        return total, recon_l, kl

# Verify architecture with dummy tensor
cfg_test = VAEConfig(in_channels=3, out_channels=1)
vae_test = ConditionalVAE(cfg_test)
dummy = tf.random.normal((2, 3, 240, 240))
r, mu, lv, z = vae_test(dummy)
print(f'✅ Architecture OK. Input {dummy.shape} → Recon {r.shape}, z {z.shape}')
del vae_test, dummy


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-03: Data Pipeline with Correct Channel Mapping
#
# FIXED from Pattie's code:
# - Uses correct channel order [T1, T1ce, T2, FLAIR] from SEG-04
# - Streams from disk via tf.data.Dataset (no RAM explosion)
# - Reads from LOCAL_DIR (Phase 1 local SSD)
# ═══════════════════════════════════════════════════════════════

def make_vae_dataset(slice_dir, target_modality, batch_size=8, shuffle=True):
    """Create a tf.data.Dataset for VAE training.
    
    target_modality: 'T1', 'T1ce', 'T2', or 'FLAIR'
        The modality to synthesize. All others are used as input.
    """
    if target_modality not in MODALITY_CHANNEL:
        raise ValueError(f'Unknown modality {target_modality}')

    target_idx = MODALITY_CHANNEL[target_modality]
    input_indices = [i for i in range(4) if i != target_idx]

    file_list = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))
    print(f'📦 {os.path.basename(slice_dir)}: {len(file_list)} slices → '
          f'synthesize {target_modality} (ch{target_idx}) from '
          f'{"+".join([k for k,v in MODALITY_CHANNEL.items() if v in input_indices])}')

    def gen():
        fs = file_list.copy()
        if shuffle:
            np.random.shuffle(fs)
        for f in fs:
            data = np.load(f)
            img = data['image'].astype(np.float32)  # (4, 240, 240)
            x = img[input_indices]                   # (3, 240, 240)
            y = img[target_idx:target_idx+1]         # (1, 240, 240)
            yield x, y

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((len(input_indices), 240, 240), tf.float32),
        tf.TensorSpec((1, 240, 240), tf.float32)
    ))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE), len(file_list)

def make_vae_joint_dataset(slice_dir, target_modalities, batch_size=8, shuffle=True):
    """Task C: synthesize multiple modalities jointly.
    target_modalities: list of modality names to synthesize (e.g. ['T1', 'T2'])"""
    target_indices = sorted([MODALITY_CHANNEL[m] for m in target_modalities])
    input_indices = [i for i in range(4) if i not in target_indices]

    file_list = sorted(glob.glob(os.path.join(slice_dir, '*.npz')))
    in_names = '+'.join([k for k,v in MODALITY_CHANNEL.items() if v in input_indices])
    out_names = '+'.join(target_modalities)
    print(f'📦 {os.path.basename(slice_dir)}: {len(file_list)} slices → '
          f'synthesize {out_names} from {in_names}')

    def gen():
        fs = file_list.copy()
        if shuffle:
            np.random.shuffle(fs)
        for f in fs:
            img = np.load(f)['image'].astype(np.float32)
            x = img[input_indices]
            y = img[target_indices]
            yield x, y

    ds = tf.data.Dataset.from_generator(gen, output_signature=(
        tf.TensorSpec((len(input_indices), 240, 240), tf.float32),
        tf.TensorSpec((len(target_indices), 240, 240), tf.float32)
    ))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE), len(file_list)

# Quick sanity check: Task A (synthesize T1ce from T1+T2+FLAIR)
test_ds, n = make_vae_dataset(SLICE_DIR_TRAIN, 'T1ce', batch_size=2)
for xb, yb in test_ds.take(1):
    print(f'\n✅ Task A batch: X {xb.shape} (T1+T2+FLAIR), Y {yb.shape} (T1ce)')
    print(f'   X range: [{xb.numpy().min():.2f}, {xb.numpy().max():.2f}]')
    print(f'   Y range: [{yb.numpy().min():.2f}, {yb.numpy().max():.2f}]')
del test_ds


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-02: Proper Training Loop (replaces Pattie's 2-slice demo)
#
# Iterates over FULL training set each epoch.
# Evaluates on validation set after each epoch.
# Tracks SSIM + PSNR for EVAL-03.
# Saves best checkpoint to Drive.
# ═══════════════════════════════════════════════════════════════

def compute_ssim_psnr(y_true, y_pred):
    """SSIM + PSNR on synthesis outputs. Inputs are channels-first.
    Transpose to channels-last for tf.image functions."""
    yt = tf.transpose(y_true, (0, 2, 3, 1))
    yp = tf.transpose(y_pred, (0, 2, 3, 1))
    # Dynamic range for z-scored data ≈ 8 (roughly [-4, +4])
    max_val = 8.0
    ssim = tf.reduce_mean(tf.image.ssim(yt, yp, max_val=max_val))
    psnr = tf.reduce_mean(tf.image.psnr(yt, yp, max_val=max_val))
    return float(ssim.numpy()), float(psnr.numpy())

def evaluate_vae(model, val_ds):
    """Compute val loss, SSIM, PSNR over validation set."""
    losses, ssims, psnrs = [], [], []
    for xb, yb in val_ds:
        recon, _, _, _ = model(xb, training=False)
        recon_loss = tf.reduce_mean(tf.math.squared_difference(yb, recon))
        losses.append(float(recon_loss.numpy()))
        s, p = compute_ssim_psnr(yb, recon)
        ssims.append(s)
        psnrs.append(p)
    return np.mean(losses), np.mean(ssims), np.mean(psnrs)

def train_vae(model, train_ds, val_ds, config, task_name, epochs=25,
              warmup_epochs=3, save_name='vae.keras'):
    """Full training loop with β-annealing + validation + checkpointing."""
    optimizer = keras.optimizers.Adam(
        learning_rate=config.learning_rate, clipnorm=1.0)
    beta_sched = BetaScheduler(beta_max=config.beta, warmup_epochs=warmup_epochs)

    best_val = float('inf')
    history = []
    save_path = os.path.join(MODEL_DIR, save_name)

    print(f'\n🚀 Training {task_name} for {epochs} epochs...')
    for epoch in range(1, epochs + 1):
        beta = beta_sched.get(epoch)
        # Reset trackers
        model.total_loss_tracker.reset_state()
        model.recon_loss_tracker.reset_state()
        model.kl_loss_tracker.reset_state()

        t0 = time.time()
        n_batches = 0
        for xb, yb in train_ds:
            model.train_step_batch(optimizer, xb, yb, beta=beta)
            n_batches += 1

        train_total = float(model.total_loss_tracker.result().numpy())
        train_recon = float(model.recon_loss_tracker.result().numpy())
        train_kl = float(model.kl_loss_tracker.result().numpy())

        val_loss, val_ssim, val_psnr = evaluate_vae(model, val_ds)
        elapsed = time.time() - t0

        log = dict(epoch=epoch, beta=beta, train_total=train_total,
                   train_recon=train_recon, train_kl=train_kl,
                   val_loss=val_loss, val_ssim=val_ssim, val_psnr=val_psnr,
                   time_s=elapsed)
        history.append(log)

        marker = ''
        if val_loss < best_val:
            best_val = val_loss
            model.save_weights(save_path.replace('.keras', '_best.weights.h5'))
            marker = ' ⭐ best'

        print(f'Ep {epoch:2d}/{epochs} | β={beta:.2f} | '
              f'train {train_recon:.4f}+{train_kl:.2f} | '
              f'val MSE={val_loss:.4f} SSIM={val_ssim:.3f} PSNR={val_psnr:.1f}dB | '
              f'{elapsed:.0f}s{marker}')

    return history

print('✅ Training infrastructure ready')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-04a: Task A — Synthesize T1ce from T1 + T2 + FLAIR
# PRESENTATION-CRITICAL. This is the result shown on 4/30.
# ═══════════════════════════════════════════════════════════════

BATCH_SIZE_VAE = 8
EPOCHS_VAE = 25

# Task A: synthesize T1ce (ch1) from T1+T2+FLAIR (ch0, ch2, ch3)
train_ds_a, n_train_a = make_vae_dataset(
    SLICE_DIR_TRAIN, 'T1ce', batch_size=BATCH_SIZE_VAE, shuffle=True)
val_ds_a, n_val_a = make_vae_dataset(
    SLICE_DIR_VAL, 'T1ce', batch_size=BATCH_SIZE_VAE, shuffle=False)

config_a = VAEConfig(in_channels=3, out_channels=1, batch_size=BATCH_SIZE_VAE)
model_a = ConditionalVAE(config_a)

history_a = train_vae(model_a, train_ds_a, val_ds_a, config_a,
                       'VAE-04a (T1ce synthesis)',
                       epochs=EPOCHS_VAE, warmup_epochs=3,
                       save_name='vae_t1ce.keras')

# Save history for EVAL-03 plots
with open(os.path.join(OUTPUT_DIR, 'vae_t1ce_history.json'), 'w') as f:
    json.dump(history_a, f, indent=2, default=float)
print('\n✅ VAE-04a complete. History saved.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-04b: Task B — Synthesize FLAIR from T1+T1ce+T2 (stretch)
# ═══════════════════════════════════════════════════════════════

train_ds_b, _ = make_vae_dataset(SLICE_DIR_TRAIN, 'FLAIR',
                                  batch_size=BATCH_SIZE_VAE, shuffle=True)
val_ds_b, _ = make_vae_dataset(SLICE_DIR_VAL, 'FLAIR',
                                batch_size=BATCH_SIZE_VAE, shuffle=False)

config_b = VAEConfig(in_channels=3, out_channels=1, batch_size=BATCH_SIZE_VAE)
model_b = ConditionalVAE(config_b)

history_b = train_vae(model_b, train_ds_b, val_ds_b, config_b,
                       'VAE-04b (FLAIR synthesis)',
                       epochs=EPOCHS_VAE, warmup_epochs=3,
                       save_name='vae_flair.keras')

with open(os.path.join(OUTPUT_DIR, 'vae_flair_history.json'), 'w') as f:
    json.dump(history_b, f, indent=2, default=float)
print('\n✅ VAE-04b complete.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-04c: Task C — Joint T1+T2 synthesis from T1ce+FLAIR (final sub)
# ═══════════════════════════════════════════════════════════════

train_ds_c, _ = make_vae_joint_dataset(
    SLICE_DIR_TRAIN, ['T1', 'T2'], batch_size=BATCH_SIZE_VAE, shuffle=True)
val_ds_c, _ = make_vae_joint_dataset(
    SLICE_DIR_VAL, ['T1', 'T2'], batch_size=BATCH_SIZE_VAE, shuffle=False)

config_c = VAEConfig(in_channels=2, out_channels=2, batch_size=BATCH_SIZE_VAE)
model_c = ConditionalVAE(config_c)

history_c = train_vae(model_c, train_ds_c, val_ds_c, config_c,
                       'VAE-04c (T1+T2 joint synthesis)',
                       epochs=EPOCHS_VAE, warmup_epochs=3,
                       save_name='vae_t1t2.keras')

with open(os.path.join(OUTPUT_DIR, 'vae_t1t2_history.json'), 'w') as f:
    json.dump(history_c, f, indent=2, default=float)
print('\n✅ VAE-04c complete.')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-05: Reconstruction Preview
# Side-by-side: input modalities | real target | synthesized target
# Saves figure for EVAL-03 and DEL-01B.
# ═══════════════════════════════════════════════════════════════

def show_synthesis(model, val_ds, target_name, n_samples=4,
                   save_name='vae_synthesis.png'):
    fig, axes = plt.subplots(n_samples, 3, figsize=(12, 3 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]

    rows_shown = 0
    for xb, yb in val_ds:
        recon, _, _, _ = model(xb, training=False)
        for i in range(xb.shape[0]):
            if rows_shown >= n_samples:
                break
            # Show one input channel (first one), real target, synth target
            in_chan = xb[i, 0].numpy()
            real = yb[i, 0].numpy()
            synth = recon[i, 0].numpy()
            axes[rows_shown, 0].imshow(in_chan, cmap='gray')
            axes[rows_shown, 0].set_title('Input (ch0)')
            axes[rows_shown, 0].axis('off')
            axes[rows_shown, 1].imshow(real, cmap='gray')
            axes[rows_shown, 1].set_title(f'Real {target_name}')
            axes[rows_shown, 1].axis('off')
            axes[rows_shown, 2].imshow(synth, cmap='gray')
            axes[rows_shown, 2].set_title(f'Synth {target_name}')
            axes[rows_shown, 2].axis('off')
            rows_shown += 1
        if rows_shown >= n_samples:
            break

    plt.tight_layout()
    out = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(out, dpi=150)
    plt.show()
    print(f'✅ Saved: {out}')

show_synthesis(model_a, val_ds_a, 'T1ce', n_samples=4,
               save_name='vae_t1ce_preview.png')


In [ ]:
# ═══════════════════════════════════════════════════════════════
# VAE-04a HANDOFF CHECKLIST (for Sipho's INT-01 pipeline)
# ═══════════════════════════════════════════════════════════════

t1ce_ckpt = os.path.join(MODEL_DIR, 'vae_t1ce_best.weights.h5')
print('═' * 60)
print('VAE-04a HANDOFF (T1ce synthesis)')
print('═' * 60)
print(f'1. Checkpoint: {t1ce_ckpt}')
print(f'2. Input:  (batch, 3, 240, 240) channels-first float32')
print(f'   Channels: [T1, T2, FLAIR] in that order')
print(f'   Output: (batch, 1, 240, 240) synthesized T1ce, linear range')
print(f'3. Dtype: float32')
print(f'4. Normalization: z-scored per-volume (matches Yue preprocessing)')
print(f'5. Inference code:')
print(f'   from <this notebook import> VAEConfig, ConditionalVAE')
print(f'   cfg = VAEConfig(in_channels=3, out_channels=1)')
print(f'   vae = ConditionalVAE(cfg)')
print(f'   vae.load_weights("{t1ce_ckpt}")')
print(f'   synth, _, _, _ = vae(x_inputs, training=False)')
print(f'6. Sanity check: vae_t1ce_preview.png')
print(f'7. Framework: TensorFlow/Keras, channels-FIRST')
print(f'   NOTE: U-Net (SEG-11) uses channels-LAST. Sipho\'s INT-01')
print(f'   must transpose VAE output before feeding to U-Net:')
print(f'      synth_cl = tf.transpose(synth, (0, 2, 3, 1))  # (B, 240, 240, 1)')
print('═' * 60)
